# Race Results Web Scraper

Scrapes 10K race results from [hubertiming.com](http://www.hubertiming.com/results/2017GPTR10K), cleans the raw HTML table into a tidy pandas DataFrame, converts finish times to minutes, and explores the results with some basic visualizations.

**Steps:**
1. Fetch and parse the page with BeautifulSoup
2. Extract table rows and headers
3. Clean the raw text into a structured DataFrame
4. Convert race times (`MM:SS` / `H:MM:SS`) into minutes
5. Explore the data with summary stats and plots

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from urllib.request import urlopen
from bs4 import BeautifulSoup

url = "http://www.hubertiming.com/results/2017GPTR10K"
html = urlopen(url)

## 2. Fetch and parse the page

In [ ]:
soup = BeautifulSoup(html, 'lxml')
type(soup)

In [ ]:
title = soup.title
print(title)

Quick look at all the links on the page, just to explore the structure.

In [ ]:
all_links = soup.find_all("a")
for link in all_links:
    print(link.get("href"))

## 3. Extract table rows

In [ ]:
rows = soup.find_all('tr')
print(rows[:10])

Before cleaning every row, check what a single row's cells look like once the HTML tags are stripped out.

In [ ]:
for row in rows:
    row_td = row.find_all('td')

str_cells = str(row_td)
cleantxt = BeautifulSoup(str_cells, "lxml").get_text()
print(cleantxt)

## 4. Clean all rows with regex

Strip HTML tags from every row using a regex pattern, and collect the cleaned text into a list.

In [ ]:
import re

list_rows = []
for row in rows:
    cells = row.find_all('td')
    str_cells = str(cells)
    clean = re.compile('<.*?>')
    clean2 = re.sub(clean, '', str_cells)  # re.sub(pattern, replacement, text)
    list_rows.append(clean2.replace("\n", ''))

print(list_rows)

## 5. Build a DataFrame from the row data

In [ ]:
df = pd.DataFrame(list_rows)
df.head(10)

In [ ]:
df1 = df[0].str.split(',', expand=True)
df1.head(10)

In [ ]:
df1[0] = df1[0].str.strip('[')
df1.head(10)

## 6. Extract table headers

In [ ]:
col_labels = soup.find_all('th')
print(col_labels)

In [ ]:
all_header = []
col_str = str(col_labels)
cleantxt2 = BeautifulSoup(col_str, "lxml").get_text()
all_header.append(cleantxt2)
print(all_header)

In [ ]:
df2 = pd.DataFrame(all_header)
df2.head()

In [ ]:
df3 = df2[0].str.split(',', expand=True)
df3.head()

## 7. Combine headers with row data

In [ ]:
frames = [df3, df1]

df4 = pd.concat(frames)
df4.head(10)

In [ ]:
df5 = df4.rename(columns=df4.iloc[0])  # use the first row as the column names
df5.head()

In [ ]:
df5.info()
df5.shape

## 8. Drop the leftover header row and any missing values

In [ ]:
df6 = df5.dropna(axis=0, how='any')
df7 = df6.drop(df6.index[0])
df7.head()  # quick way to peek at the data without printing the whole thing

## 9. Clean up column names and values

In [ ]:
df7.rename(columns={'[Place': 'Place'}, inplace=True)  # inplace=True modifies df7 directly instead of returning a copy
df7.rename(columns={' Team]': 'Team'}, inplace=True)
df7.head()

In [ ]:
df7['Team'] = df7['Team'].str.strip(']')
df7.head()

In [ ]:
df7.columns = df7.columns.str.strip()

In [ ]:
df7['Time']

## 10. Convert race times to minutes

Some times are `MM:SS` and others are `H:MM:SS` (for slower finishers past the 1-hour mark), so the split handles both formats.

In [ ]:
time_list = df7['Time'].tolist()
time_mins = []
for i in time_list:
    parts = i.split(':')
    if len(parts) == 2:
        m, s = parts
        math = (int(m) * 60 + int(s)) / 60
    elif len(parts) == 3:
        h, m, s = parts
        math = (int(h) * 3600 + int(m) * 60 + int(s)) / 60
    time_mins.append(math)
print(time_mins)

In [ ]:
df7['Runner_mins'] = time_mins
df7.head()

## 11. Exploratory data analysis

In [ ]:
df7.describe(include=[np.number])  # only numerical columns

In [ ]:
from pylab import rcParams
rcParams['figure.figsize'] = 15, 5
df7.boxplot(column='Runner_mins')
plt.grid(True, axis='y')
plt.ylabel('Time')
plt.xticks([1], ['Runners'])

In [ ]:
x = df7['Runner_mins']
ax = sns.distplot(x, hist=True, kde=True, rug=False, color='g', bins=25, hist_kws={'edgecolor': 'black'})
plt.show()

Compare the finish-time distributions for female vs. male runners.

In [ ]:
f_fuko = df7.loc[df7['Gender'] == ' F']['Runner_mins']
m_fuko = df7.loc[df7['Gender'] == ' M']['Runner_mins']
sns.distplot(f_fuko, hist=True, kde=True, rug=False, hist_kws={'edgecolor': 'black'}, label='Female')
sns.distplot(m_fuko, hist=False, kde=True, rug=False, hist_kws={'edgecolor': 'black'}, label='Male')
plt.legend()

In [ ]:
g_stats = df7.groupby("Gender", as_index=True).describe()
print(g_stats)

In [ ]:
df7.boxplot(column='Runner_mins', by='Gender')
plt.ylabel('Time')
plt.suptitle("")